# M3L3 E03 — Orquestador básico
### Módulo 3 · Lecture 3 · Sistemas Multiagente

## ¿Qué vas a aprender hoy?
- combinar clasificador, router y agentes.
- Conectar el concepto con M3L2.
- Leer código pequeño con explicación previa.
- Interpretar resultados y checks.


## ¿Qué necesitás saber antes?

Venís de M3L2 con LangChain, LCEL, PromptTemplate, RAG con FAISS y memoria conversacional. En M3L3 usamos esas piezas para coordinar varios agentes.

> **Sistema multiagente:** arquitectura donde varias unidades especializadas colaboran bajo una política de coordinación.


## Instalación e imports

En un notebook productivo podrías instalar `langchain`, `langchain-openai` y `faiss-cpu`. Aquí usamos Python estándar para que el foco sea el diseño multiagente y no la API key.


In [ ]:
from typing import Callable, TypedDict, Literal
from dataclasses import dataclass, field
import json

print("Setup listo: usamos Python estándar para que el notebook pueda correr sin API key.")


## Sección 1 — Orquestar no es responder

> **Orquestador:** componente que decide el flujo. Clasifica, elige agente y ejecuta, pero no necesita ser experto.

```text
usuario -> clasificador -> registro de agentes -> especialista -> respuesta
```


Cargamos conocimiento, clasificador y agentes mínimos para que el notebook sea autónomo.


In [ ]:
knowledge_base = {
    "hr": [
        "Vacaciones: cada empleado tiene 15 días hábiles por año.",
        "Beneficios: el seguro médico inicia el primer día de trabajo.",
        "Licencias: registrar el pedido en PeopleOps y avisar al manager.",
    ],
    "tech": [
        "VPN: reiniciar el cliente, validar MFA y abrir ticket si persiste.",
        "Contraseña: restablecer desde el portal de identidad.",
        "Notebook: reportar equipo dañado con número de serie.",
    ],
    "billing": [
        "Facturas: cargar comprobantes antes del día 25.",
        "Reembolsos: adjuntar recibo, monto y centro de costo.",
        "Pagos: se procesan los viernes por la tarde.",
    ],
}

KEYWORDS = {
    "hr": ["vacaciones", "beneficio", "seguro", "licencia"],
    "tech": ["vpn", "contraseña", "mfa", "notebook"],
    "billing": ["factura", "facturas", "reembolso", "pago", "recibo"],
}

def detect_domains(query: str) -> list[str]:
    text = query.lower()
    matches = [domain for domain, words in KEYWORDS.items() if any(word in text for word in words)]
    return matches or ["unknown"]

def retrieve(domain: str, query: str, k: int = 2) -> list[str]:
    docs = knowledge_base[domain]
    query_words = set(query.lower().replace("¿", "").replace("?", "").split())
    def score(doc: str) -> int:
        return sum(1 for word in query_words if word.strip(",.") in doc.lower())
    return sorted(docs, key=score, reverse=True)[:k]

def make_agent(domain: str, name: str) -> Callable[[str], str]:
    return lambda query: f"{name}: " + " | ".join(retrieve(domain, query, 2))
AGENTS = {"hr": make_agent("hr", "HRAgent"), "tech": make_agent("tech", "TechAgent"), "billing": make_agent("billing", "BillingAgent")}


## Sección 2 — Trace y `handle_query`

El trace muestra `[Thought]`, `[Action]` y `[Observation]`. Sirve para auditar decisiones sin leer código interno.


In [ ]:
# TODO: completar route y handle_query.
def print_trace(step: str, content: str):
    print(f"[{step}] {content}")

def route(intent: str) -> Callable[[str], str] | None:
    return None

def handle_query(query: str) -> str:
    return "TODO: clasificar, rutear y ejecutar"


## Sección 3 — Pruebas del orquestador

Usamos consultas de HR, Tech y Billing para mirar el trace completo.


In [ ]:
for q in ["vacaciones", "vpn", "factura", "almuerzo"]:
    print("\nQUERY", q)
    print(handle_query(q))


## Checks automáticos

Los checks verifican el contrato mínimo del ejercicio. En Starter pueden fallar hasta completar los TODOs; en Resolution deben pasar.


In [ ]:
def run_checks():
    assert isinstance(handle_query("vpn"), str)
    print("Checks E03 OK")
run_checks()


## ¿Qué aprendiste hoy?

- Combinar clasificador, router y agentes.
- Separar responsabilidades vuelve el sistema más auditable.
- Los contratos explícitos hacen que el orquestador dependa menos de texto libre.

## Próximo ejercicio

Continuá con el siguiente notebook de M3L3 para agregar una pieza más de coordinación multiagente.
